colab連結:https://colab.research.google.com/drive/14q8vRg8eKzjzdVoT3dDvsQmNzB1YakvC?usp=sharing

姓名:鄭若岑

學號:112403535

**python:3.12.13**

**requirements.txt :**

pandas

folium

langchain

langchain-community

langchain-core

langchain-google-genai

google-generativeai

pypdf

faiss-cpu

pydantic

ipython

**專案動機:**

現行長照機構查詢大多使用選單式的查詢機制，使用者必須事先清楚行政區劃以及標準的服務名稱（如：喘息服務、輔具服務）才能進行篩選。

然而，對於一般家屬或長輩而言，他們往往只有模糊的訴求（例如:照顧壓力太大想休息幾天等）。本專案旨在開發一個智慧導航Agent，讓使用者透過自然語言表達需求，並由AI自動解析地點與需求類型，提供符合使用者需求的機構並顯示在地圖上，降低長照資源的檢索門檻。

**資料及來源**:

政府資料開放平台 [長照ABC據點](https://data.gov.tw/dataset/88270) [abc.csv](https://ltcpap.mohw.gov.tw/publish/abc.csv) 用於查詢機構

衛福部長照十年計畫2.0 [長照2.0核定本](https://www.mohw.gov.tw/dl-78115-5511ccc0-cae0-4d16-b729-6d0e16228fb5.html)用於建立RAG，當訴求較模糊(或是沒有說出具體所需的服務名稱)時，可查詢服務內容及分級定義

**區塊一**

安裝依賴套件: 包含 LangChain 相關組件、PDF 解析工具及向量搜尋引擎。

配置 API 金鑰: 安全地匯入 Google Gemini API Key 以啟動模型。

模型初始化: 設定 LLM 參數（如 temperature=0.2 以確保回覆的穩定與精確）並載入 Embedding 嵌入模型。

下載所需檔案

In [1]:
#區塊1
# 1.安裝套件
!pip install -q -U langchain langchain-community langchain-google-genai google-generativeai pypdf faiss-cpu pandas
import folium
from IPython.display import display
import os
import getpass
import pandas as pd
import requests

# 2.輸入 Google Gemini API Key
if "GOOGLE_API_KEY" not in os.environ:
    print("請輸入您的 Google Gemini API Key:")
    os.environ["GOOGLE_API_KEY"] = getpass.getpass()

from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.2)
from langchain_google_genai import GoogleGenerativeAIEmbeddings
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")

import os
import requests

def download_file(url, save_name, is_retry=False):
    if not os.path.exists(save_name):
        print(f"正在嘗試下載 {save_name}...")
        try:
            headers = {'User-Agent': 'Mozilla/5.0'}
            response = requests.get(url, headers=headers, stream=True, timeout=60)
            response.raise_for_status()

            with open(save_name, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
            print(f"{save_name} 下載完成！")

        except Exception as e:
            if not is_retry:
                print(f"{save_name} 官方路徑異常、連線逾時：{e}")
                github_backup_url = f"https://raw.githubusercontent.com/namonroll/long_term_care/main/{save_name}"
                print(f"切換至 GitHub 備援路徑下載...")

                download_file(github_backup_url, save_name, is_retry=True)
            else:
                print(f"錯誤：備援路徑下載失敗。")
    else:
        print(f"{save_name} 已存在，跳過下載。")

csv_url = "https://ltcpap.mohw.gov.tw/publish/abc.csv"
pdf_url = "https://www.mohw.gov.tw/dl-78115-5511ccc0-cae0-4d16-b729-6d0e16228fb5.html"

download_file(csv_url, "abc.csv")
download_file(pdf_url, "1051219長照2.0核定本.pdf")
print("環境設定與 API 初始化完成")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.5/112.5 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.

**區塊2**

**1.資料清洗:**

移除原始 CSV 中可能殘留的引號與空格。

針對「地址」、「服務項目」與「ABC分級」欄位進行格式統一，確保搜尋時不會因為型態錯誤而失效。

**2.資料過濾:**

使用Pandas的mask機制進行篩選。

地點過濾:透過str.contains同時鎖定縣市與行政區。

服務映射:根據LLM判斷出的關鍵字，動態搜尋機構特約項目。

級別篩選:支援A、B、C級別的清單匹配。

**3.輸出:**

文字回傳:生成易於閱讀的機構清單，供LLM整合進最終回覆。

DataFrame回傳:保留原始結構化資料（含經緯度），作為後續互動式地圖渲染的輸入源。

In [2]:
#區塊2
#讀取 CSV 檔
csv_file_path = "abc.csv"
df = pd.read_csv(csv_file_path)

# 清理欄位名稱 (防範引號殘留)
df.columns = [c.replace('"', '').strip() for c in df.columns]

# 2.過濾函數
def find_local_agencies(city, district, service_keyword="", abc_levels=None):
    df['地址全址'] = df['地址全址'].astype(str)
    df['特約服務項目'] = df['特約服務項目'].astype(str)
    df['O_ABC'] = df['O_ABC'].astype(str).str.strip()

    mask = df['地址全址'].str.contains(city, na=False) & df['地址全址'].str.contains(district, na=False)

    if service_keyword:
        mask = mask & df['特約服務項目'].str.contains(service_keyword, na=False)

    if abc_levels and isinstance(abc_levels, list) and len(abc_levels) > 0:
        mask = mask & df['O_ABC'].isin(abc_levels)

    filtered_df = df[mask]
    # 上限調高到10筆
    results = filtered_df.head(10)

    level_msg = f"{'、'.join(abc_levels)}級" if abc_levels else "任何級別"

    if results.empty:
        # 找不到時DataFrame回傳None
        return f"目前在 {city}{district} 找不到提供「{service_keyword}」的 {level_msg} 機構。", None

    agency_info = f"找到以下 {level_msg} 機構：\n"
    for _, row in results.iterrows():
        name = str(row['機構名稱']).replace('"', '')
        addr = str(row['地址全址']).replace('"', '')
        service = str(row['特約服務項目']).replace('"', '')
        abc_type = str(row['O_ABC']).replace('"', '')
        phone = str(row['機構電話']).replace('"', '')

        agency_info += f"- 機構名稱：{name} ({abc_type}級)\n"
        agency_info += f"  地址：{addr}\n"
        agency_info += f"  電話：{phone}\n"
        agency_info += f"  提供服務：{service}\n\n"

    return agency_info, results

**區塊3**

**1.資料類型校準:**

使用 pd.to_numeric 強制轉換經緯度欄位，並利用 errors='coerce' 自動處理 CSV 中可能存在的異常字元。

執行 dropna 剔除座標缺失的機構，確保地圖渲染時不會發生程式崩潰。

**2.動態視角縮放:**

系統會自動計算所有搜尋結果的平均經緯度，將地圖中心點自動聚焦在機構最密集的區域，省去使用者手動調整地圖的麻煩。

**3.互動UI:**

大頭針: 使用綠色標示點並附帶info-sign圖示。

懸浮提示: 游標移過時即時顯示機構名稱與 ABC 分級，實現「快速預覽」。

彈出式視窗:點擊大頭針可查看格式化的詳細資訊（含電話與特約項目），模擬真實 Web App 的體驗。

In [3]:
# 區塊3
def create_interactive_map(agencies_df):
    """
    根據傳入的機構資料表，生成包含大頭針與 Hover 效果的 Folium 地圖
    """
    if agencies_df is None or agencies_df.empty:
        return None

    #確保經緯度是數字格式
    agencies_df['緯度'] = pd.to_numeric(agencies_df['緯度'], errors='coerce')
    agencies_df['經度'] = pd.to_numeric(agencies_df['經度'], errors='coerce')

    #剔除經緯度無效的資料
    agencies_df = agencies_df.dropna(subset=['緯度', '經度'])

    if agencies_df.empty:
        return None

    #計算地圖的中心點(取所有機構的平均經緯度)
    center_lat = agencies_df['緯度'].mean()
    center_lon = agencies_df['經度'].mean()

    #建立基礎地圖
    m = folium.Map(location=[center_lat, center_lon], zoom_start=15, width=600, height=400)

    #把每一個機構畫上地圖
    for _, row in agencies_df.iterrows():
        lat = row['緯度']
        lon = row['經度']
        name = str(row['機構名稱']).replace('"', '')
        abc_type = str(row['O_ABC']).replace('"', '')
        service = str(row['特約服務項目']).replace('"', '')
        phone = str(row['機構電話']).replace('"', '')

        # Hover效果(游標移過去顯示文字)
        hover_text = f"{name} ({abc_type}級)"

        # 點擊效果(點擊彈出詳細資訊)
        popup_html = f"""
        <div style="width: 200px;">
            <b>{name}</b><br>
            <span style="color: blue;">級別：{abc_type}級</span><br>
            📞 電話：{phone}<br>
            📋 服務：{service}
        </div>
        """

        # 加入大頭針
        folium.Marker(
            location=[lat, lon],
            tooltip=hover_text, #Hover效果
            popup=folium.Popup(popup_html, max_width=300), #點擊效果
            icon=folium.Icon(color='green', icon='info-sign')
        ).add_to(m)

    return m

**區塊4**

**RAG 知識庫建置 (PDF 向量化)**


處理「長照 2.0 核定本」PDF 文件。由於政策文件內容龐大且非結構化，使用RAG，將其轉化為可供語意檢索的向量資料庫，以解決LLM缺少特定政策知識或產生幻覺的問題。

**1.文件解析與負載:**

使用 PyPDFLoader 解析 PDF 內容，將原始文本提取為可處理的 Document 物件。

**2.文本切割:**

採用RecursiveCharacterTextSplitter將長文本切分為800字的Chunks。

設定chunk_overlap=100以確保跨區塊的語意連續性，避免資訊在切割處斷開。

**3.防斷線批次嵌入:**

批次處理:由於Embedding API存在頻率限制，使用batch_size=10的機制進行分段上傳。

指數退避與重試:捕捉 429 Resource Exhausted 錯誤。當觸發 API 限制時，系統會自動「強制休息」並重試，確保大型文件能穩定地完成向量化。

**4.向量數據庫 (FAISS):**

使用Facebook AI開發的FAISS。

透過 Google 的 Embedding 模型將文字轉為高維度向量，實現基於「語意」而非「關鍵字」的精準檢索。



In [6]:
#區塊4
import os
from langchain_community.vectorstores import FAISS

save_path = "faiss_index_care"

# --- 判斷邏輯：有存檔就讀取，沒存檔才建立 ---
if os.path.exists(save_path):
    print(f"偵測到本地索引庫 '{save_path}'，正在直接載入...")
    vectorstore = FAISS.load_local(save_path, embeddings, allow_dangerous_deserialization=True)
    print("載入成功！")
else:
    print(f"找不到本地索引，開始從 PDF 重新建立 (這可能需要幾分鐘)...")

    loader = PyPDFLoader(pdf_file_path)
    pages = loader.load()
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
    docs = text_splitter.split_documents(pages)

    # 建立向量庫
    batch_size = 10
    vectorstore = None
    for i in range(0, len(docs), batch_size):
        batch = docs[i : i + batch_size]
        if vectorstore is None:
            vectorstore = FAISS.from_documents(batch, embeddings)
        else:
            vectorstore.add_documents(batch)
        print(f"已處理進度: {min(i + batch_size, len(docs))}/{len(docs)}")
        if i + batch_size < len(docs): time.sleep(10)

    # 建立完後立刻存檔到本地，下次就不用重跑！
    vectorstore.save_local(save_path)
    print(f"索引已存檔至 '{save_path}'")

# 建立檢索器
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print("檢索器已就緒！")

🔍 找不到本地索引，開始從 PDF 重新建立 (這可能需要幾分鐘)...
已處理進度: 10/239
已處理進度: 20/239
已處理進度: 30/239
已處理進度: 40/239
已處理進度: 50/239
已處理進度: 60/239
已處理進度: 70/239
已處理進度: 80/239
已處理進度: 90/239
已處理進度: 100/239
已處理進度: 110/239
已處理進度: 120/239
已處理進度: 130/239
已處理進度: 140/239
已處理進度: 150/239
已處理進度: 160/239
已處理進度: 170/239
已處理進度: 180/239
已處理進度: 190/239
已處理進度: 200/239
已處理進度: 210/239
已處理進度: 220/239
已處理進度: 230/239
已處理進度: 239/239
索引已存檔至 'faiss_index_care'
檢索器已就緒！


**區塊5**

**1.動態檢索增強:**

系統檢索原始問題，根據解析出的ABC分級動態修改檢索字串。確保FAISS向量庫能精準抓取PDF裡關於該特定級別的法規定義。

**2.資料融合:**

空間維度: 呼叫 create_interactive_map 生成 Folium 地圖物件。

政策維度: 擷取 RAG 檢索到的政策片段。

實體維度: 導入 CSV 篩選後的真實機構清單。

**3.提示詞工程:**

角色設定: 將 LLM 定義為「溫暖、專業的台灣長照導航員」。

白話文翻譯: 規定系統必須將生硬的政策術語（如：C 級巷弄長照站）轉化為家屬聽得懂的語言（如：鄰里間的活動據點）。

防錯與引導: 當資料庫無匹配結果時，設定「安撫與導流」機制（建議撥打 1966 專線），確保使用者不會得到死胡同式的回答。

In [7]:
#區塊5
from langchain_core.prompts import PromptTemplate

def long_term_care_agent(user_query, target_city, target_district, service_keyword="", abc_levels=None):

    level_str = f" ({'、'.join(abc_levels)}級)" if abc_levels else " (不限級別)"
    print(f"系統處理中：分析 {target_city}{target_district}{level_str} 的「{service_keyword}」需求...\n")

    # 1. 從 CSV 取得當地機構資訊 (加入 ABC 條件)
    local_agencies_text, local_agencies_df = find_local_agencies(target_city, target_district, service_keyword, abc_levels)

    # 2. 生成地圖
    agency_map = create_interactive_map(local_agencies_df)
    # 2.強化 RAG 檢索邏輯
    # 如果使用者有指定ABC級別，把這個關鍵字加到檢索字串中，
    # 確保FAISS向量庫能抓到 PDF 裡面關於該級別的定義。
    rag_query = user_query
    if abc_levels:
        rag_query += f" 請詳細說明長照 {', '.join(abc_levels)} 級單位是什麼？提供哪些功能？"

    retrieved_docs = retriever.invoke(rag_query)
    policy_context = "\n\n".join([doc.page_content for doc in retrieved_docs])

    # 3. 組合Prompt(特別強調解釋ABC分級)
    prompt_template = """
    你是一位專業、溫暖且富有同理心的「台灣長照 2.0 導航員」。
    請根據以下提供的【政策參考資料】與【當地機構清單】，來回答民眾的問題。

    民眾的問題：{query}

    【政策參考資料】(來自長照2.0核定本)：
    {context}

    【當地機構清單】(來自資料庫精確篩選)：
    {agencies}

    回答指引：
    1. 語氣要像是在跟長輩或家屬對話，溫暖且清晰易懂。
    2. 先回答民眾問題。如果民眾的問題或機構清單中有提到「A級、B級、C級」，請務必利用【政策參考資料】用白話文解釋這些級別代表什麼意思（例如B級通常是提供日常居家服務）。
    3. 接著，明確列出【當地機構清單】給民眾參考。
    4. 如果清單找不到機構，請安撫民眾並建議他們撥打 1966 長照專線。
    """

    prompt = PromptTemplate(
        input_variables=["query", "context", "agencies"],
        template=prompt_template
    )

    # 4. 呼叫 LLM
    chain = prompt | llm
    response = chain.invoke({
        "query": user_query,
        "context": policy_context,
        "agencies": local_agencies_text
    })

    return response.content,agency_map


**區塊6**

**1.結構化參數萃取**
技術實現: 利用 Pydantic 定義 CareQuery 類別，並使用 with_structured_output 限制 LLM 的回傳格式。

在地化處理: 強制將縣市名稱校正為「臺」字開頭，並自動將非標準地標（如：忠貞市場）轉換為行政區（如：中壢區）。

**2.動態檢索策略**
系統會根據第一階段的解析結果，動態決定執行路徑：

快速路徑: 若使用者需求明確（如：提到「送餐」、「輔具」），系統直接識別服務關鍵字，跳過 RAG 檢索以節省 API 成本與時間。

RAG 補完路徑: 若使用者需求模糊（如：提到「照顧到好累」、「怕老伴跌倒」），系統會帶著原話進入 RAG 向量庫 檢索政策定義，由 LLM 二次判定最符合的標準服務（如：喘息服務或輔具申請）。

**3.多模態回傳與渲染**
解構式回傳: 函數同時回傳 LLM 生成的「溫馨回覆」文字與 Folium 生成的「互動式地圖」。

追問機制: 內建地點檢查邏輯，若關鍵位置資訊缺失，會主動引導使用者補充，確保後續 CSV 搜尋的準確性。

In [8]:
#區塊6
from IPython.display import display, clear_output # 加入 clear_output
from pydantic import BaseModel, Field
from typing import List, Optional
from typing import Literal

# 1. 定義我們希望 LLM 抓取的「資料結構」
class CareQuery(BaseModel):
    target_city: Literal[
        "", "臺北市", "新北市", "桃園市", "臺中市", "臺南市", "高雄市",
        "基隆市", "新竹縣", "新竹市", "苗栗縣", "彰化縣", "南投縣",
        "雲林縣", "嘉義縣", "嘉義市", "屏東縣", "宜蘭縣", "花蓮縣",
        "臺東縣", "澎湖縣", "金門縣", "連江縣"
    ] = Field(description="台灣縣市名稱，必須轉換為『臺』字開頭的標準名稱。若未提到則為空字串。")
    target_district: str = Field(description="台灣鄉鎮市區名稱，需包含區、鎮或鄉，例如：三民區、溪湖鎮。如果沒有提到，請輸出空字串。")
    service_keyword:Literal[
        "", "輔具及居家無障礙環境改善服務"
          ,"喘息服務"
          ,"巷弄長照站"
          ,"居家服務"
          ,"專業照護服務"
          ,"日間照顧服務(BB)"
          ,"居家失能個案家庭醫師照護服務"
          ,"個案管理服務"
          ,"社區式交通接送服務(BD03)"
          ,"家庭托顧服務(BC)"
          ,"營養餐飲服務"
          ,"交通接送服務(DA01)"
          ,"專業照護服務(CC01)"
          ,"小規模多機能服務"
          ,"交通接送服務"
          ,"到宅沐浴車服務"
          ,"團體家屋服務"
    ] = Field(description="使用者需要的長照服務關鍵字，例如：居家、喘息、營養餐飲、輔具。如果沒有明確提到，請輸出空字串。")
    abc_levels: Optional[Literal['A', 'B', 'C', '']] = Field(
        default="",
        description="依據使用者提出的機構類型：A-社區整合型服務中心，B-複合型服務中心，C-巷弄長照站。若無需求則輸出空字串或 None。"
    )


# 2.建立一個專門用來萃取參數的 LLM
# 將定義的格式綁定到 Gemini 上
extractor_llm = llm.with_structured_output(CareQuery)

# 3.建立最終的聊天介面函數
def chat_with_agent_advanced(user_input):
    print("\n [階段一：初步解析] 正在捕捉您的位置與明確需求...")
    initial_info = extractor_llm.invoke(user_input)

    # 攔截機制：如果不知道在哪裡，系統無法查 CSV，必須先追問
    if not initial_info.target_city or not initial_info.target_district:
        return " 導航員：聽起來您非常需要協助！為了幫您找到最適合的機構，可以請您先告訴我您所在的「縣市」與「行政區」嗎？（例如：桃園市中壢區）", None

    # 取得第一階段LLM自己判斷的結果
    refined_service = initial_info.service_keyword

    # 如果LLM抓不到明確服務（回傳空字串），才啟動RAG
    if not refined_service:
        print(" [階段二：政策檢索] 需求較模糊，正在翻閱長照 2.0 政策文件尋找解答...")

        # 帶著使用者的原話去PDF找關聯段落
        policy_docs = retriever.invoke(user_input)
        policy_context = "\n\n".join([doc.page_content for doc in policy_docs])

        #建立一個專門用來「決定服務名稱」的Prompt
        decision_prompt_template = """
        你是一位長照需求評估專員。請根據以下【長照政策文件】，判斷使用者的這句話，最符合我們資料庫中的哪一項標準服務？

        使用者原話：{user_input}

        【長照政策文件參考】(請依此判斷，例如文件中提到減輕照顧者負擔通常是喘息服務)：
        {context}

        請務必「只」從以下清單中選擇最符合的一項輸出（不要加上任何其他文字或標點符號）：
        居家、喘息、營養餐飲、輔具、交通接送、日間照顧

        如果文件中找不到任何關聯，請輸出空字串。
        """

        decision_prompt = PromptTemplate(
            input_variables=["user_input", "context"],
            template=decision_prompt_template
        )

        # 呼叫LLM進行二次服務判定
        service_chain = decision_prompt | llm
        refined_service = service_chain.invoke({
            "user_input": user_input,
            "context": policy_context
        }).content.strip()
        print(f"[階段三：語意判定] 根據法規，將您的需求定義為：'{refined_service}'")

    else:
        print(f"[階段二：語意判定] 您的需求很明確，系統直接辨識為：'{refined_service}' (跳過文件檢索以加速)")

    print(f"[最終大腦參數] 地點: {initial_info.target_city}{initial_info.target_district} | 服務: '{refined_service}' | 級別: {initial_info.abc_levels}\n")

    # 正式呼叫主程式，去查CSV並生成最終回覆
    return long_term_care_agent(
        user_query=user_input,
        target_city=initial_info.target_city,
        target_district=initial_info.target_district,
        service_keyword=refined_service,
        abc_levels=initial_info.abc_levels
    )

# =====測試區=====

print("========== 長照 2.0 智慧導航員已上線 ==========")
print("您可以開始輸入問題了。輸入 'exit' 結束對話。\n")

while True:
    user_msg = input("您（輸入 exit 結束）：")

    if user_msg.lower() in ['exit', 'quit']:
        print("導航員：祝您平安順心！再見。")
        break

    if not user_msg.strip():
        continue

    response_text, response_map = chat_with_agent_advanced(user_msg)


    if response_map is not None:
        clear_output(wait=True)
        print("\n導航員回覆：")
        print("-" * 50)
        print(response_text)
        print("=" * 60 + "\n")
        print("【機構地圖】 (點擊圖標以查看詳細資訊)")
        display(response_map)
    else:
        print("\n導航員回覆：")
        print("-" * 50)
        print(response_text)
        print("=" * 60 + "\n")


導航員回覆：
--------------------------------------------------
您好！非常理解您想為家人尋找合適的喘息服務，這份心意很讓人感動。您提到想找平鎮區有喘息服務的A級機構，讓我來為您說明一下長照2.0的服務體系，並提供您相關資訊。

在長照2.0的服務體系中，我們將服務單位分為A、B、C三個級別，它們各自扮演著不同的角色：

*   **A級－社區整合型服務中心 (就像是長照服務的「總指揮中心」或「導航員」)：**
    A級單位主要負責幫您規劃照顧計畫，並連結、協調您所在區域（例如平鎮區）內各種長照資源。它會確保您能順利獲得所需的服務，並提升區域內的服務能量。A級單位本身必須同時提供「日間照顧」和「居家服務」，並且會擴充提供至少一項其他服務，例如營養餐飲、居家護理、居家/社區復健、**喘息服務**或輔具服務等。所以，A級單位是幫您「整合」服務的關鍵，確保您的照顧計畫能被完整落實。

*   **B級－複合型服務中心 (可以想像成是長照服務的「複合式服務站」)：**
    B級單位是直接提供多樣化在地照顧服務的主力。它提供的服務非常多元，像是居家服務、日間照顧、家庭托顧、營養餐飲、交通接送，當然也包含您需要的「喘息服務」、輔具租借、居家護理等等。我們平時接觸到的許多長照機構、護理之家、日照中心等，很多都屬於B級單位，它們是實際提供服務、讓您直接獲得照顧的據點。

*   **C級－巷弄長照站 (則是離家最近的「巷口柑仔店」)：**
    C級單位提供的是具近便性的服務，就像您家巷口的柑仔店一樣方便。它提供短時數的照顧服務、臨托服務（這也是一種短期的喘息服務）、營養餐飲服務（共餐或送餐），以及讓長輩就近參與社會活動的場所。

---

根據您想找「平鎮有喘息服務的A級機構」的需求，我們在資料庫中篩選後，目前平鎮區直接提供喘息服務的機構，大多屬於B級單位。雖然A級單位也能提供喘息服務，但它的主要功能是整合與協調，而B級單位則是直接提供服務的主力。

以下這些位於平鎮區的B級機構都有提供喘息服務，您可以參考看看：

*   **桃園市私立瑞生老人長期照顧中心(養護型)**
    *   地址：桃園市平鎮區金陵路5段32-2號
    *   電話：03-4502966
    *   提供服務：喘息服務

*   *

KeyboardInterrupt: Interrupted by user

**matrix**

| | 測試類型 | 測試指令 |回覆結果| 地點正確 | 服務正確 | 分級正確 |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- |
|1|需求明確|我想在桃園市中壢區找有提供喘息服務的B型機構|精準提供|✅️|✅️|✅️|
|2|需求明確|我想在大安區找有提供交通接送服務的C型機構|精準提供|✅️|✅️|✅️|
|3|需求明確|我想在嘉義市東區找有提供營養餐服務的A型機構|精準提供|✅️|✅️|✅️|
|4|需求明確|我想在大安區找有提供接送服務的C型機構|精準提供|✅️|✅️|✅️|
|5|地點模糊|我想在新明國中附近找有提供喘息服務的B型機構|精準提供|✅️|✅️|✅️|
|6|地點模糊|我想在虎頭山附近找有提供接送服務的C型機構|追加詢問地點後精準提供|✅️|✅️|✅️|
|7|地點模糊|我想找有提供營養餐服務的A型機構|追加詢問地點後精準提供|✅️|✅️|✅️|
|8|地點模糊|我想找忠貞市場附近有提供接送服務的C型機構|精準提供|✅️|✅️|✅️|
|9|服務模糊|我在三民區，有幾天需要找人幫忙照護|精準提供|✅️|✅️(喘息服務)|✅️(無分級)|
|10|服務模糊|我想在三峽找C型機構，沒辦法自行前往|精準提供|✅️|✅️(交通接送服務)|✅️|
|11|服務模糊|我想在三峽找C型機構，平常幫他清潔比較不方便，也不方便出門，想找人幫忙|預期出現到宅沐浴車，但系統判斷為居家服務也可行|✅️|⚠️(居家服務)|✅️|
|12|服務模糊|我想在萬華找有餐飲的A型機構|精準提供|✅️|✅️|⚠️該區無A型營養餐飲服務，搜尋B型|
|13|要求不明|我找平鎮的社區型機構|查詢RAG後提供篇居家、巷弄或社區，服務不限的機構|✅️|✅️(無要求)|✅️|
|14|要求不明|我現在真的很想睡覺很累鹿港好遠|預期出現喘息服務或交通接送，查詢RAG後提供交通接送服務|✅️|✅️部分正確(交通接送)|⚠️無特別要求但出現分級解釋B型|
|15|要求不明|花生醬真好吃|查詢RAG後提供日間照顧服務機構|✅️|✅️(無對錯之分)|⚠️無特別要求但出現分級解釋B型|

--------------------------------------------------------------------------------------------------------
**測試評估:**


1.當需求明確時通常都能正確提供相關機構，尤其在地點方面幾乎沒問題，少部分在無法辨識地點時追問後也回答正確。

2.服務辨識方面，在服務需求較模糊時也能判斷，偶爾會出現和預期回答有點偏差的服務判斷，但同樣能滿足使用者要求。在要求不明時會直接亂湊，將來可以擴展實作追問詳細需求。

3.在分級沒特別要求時偶爾會亂塞B級，將來改善可多測試不同discription或增加追問功能。

**發現與結論**

**1.核心發現：語意翻譯的力量**

從口語到法規的橋樑：測試顯示，LLM 在處理「非標準訴求」時表現優異。使用者不需要記住繁瑣的「交通接送服務（DA01）」等代碼，只需表達「出門不方便」或「想找車載爸爸」，系統即可透過語意映射精準對接 CSV 資料庫。

地理魯棒性：系統能辨識「新明國中」、「忠貞市場」等非行政區地標，並成功將其轉化為空間查詢座標，這對在地家屬具有實用價值。

**2.技術評估：RAG 與 結構化輸出的協同**

動態路由的效率：引入「需求明確度判斷」後，系統能在簡單需求下實現回應；在複雜需求下則啟動 RAG 檢索，確保了解釋的權威性，平衡了API成本與回答深度。

結構化約束：強制 LLM 輸出符合資料庫格式的JSON，顯著降低了早期開發中常見的「格式幻覺」問題，使CSV過濾的成功率提升。

**3.侷限性與改進空間**

過度推論：在測試第 15 項「花生醬」案例中發現，當輸入完全無關時，AI 仍會嘗試「硬湊」答案。這顯示未來需要引入 「信心門檻」 機制，學會適時「拒絕回答」或「主動追問」。

服務重疊性：對於如「到宅沐浴」與「居家服務」的細微差異，僅靠 PDF 檢索有時仍會產生判斷重疊，未來可透過更精細的 Prompt 特徵工程來優化。

**未來展望**

即時 API 串接：目前採用靜態 CSV，未來若能對接機構即時資訊API，將可提供即時的機構餘額與預約狀況。

多輪對話記憶：讓 Agent 記住使用者家中的具體狀況（如：家有失智症長輩），在後續對話中能提供更具延續性的個性化建議。

多模態輸入：整合語音辨識，讓不擅長打字的長輩也能透過口述直接與「導航員」溝通。